# **MetaPassAV:** EDA for Avatars

_uses pre-processed data_  

Due to the limited dataset:  

> **Window Size:** 11 (appx. 44 sec / sample)  
> **Overlap:** 1 (step)

### Libraries and Constants

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import pickle

In [ ]:
W = 11  # window size
OVERLAP = 9 
STRIDE = W - OVERLAP

---

## Data-Handling

In [ ]:
DIR_PREFIX = "./"
CLEAN_DATA_PATH = f'{DIR_PREFIX}data/'
MODEL_PATH = f'{DIR_PREFIX}models/'
MODEL_DATA_PATH = f'{CLEAN_DATA_PATH}/raw_split'

In [4]:
df = pd.read_csv(f'{CLEAN_DATA_PATH}AvatarBeh_FeatureRich.csv')
df.shape

(4121, 31)

In [5]:
df.head()

,SessionID,StartTime,TotalTimeSpent,DistanceFromStart,DistanceFromPrevious,DistanceSoFar,AngleChange,bearing_elevation,AngularShift,LinVelocity,...,Dir_S,image1,image2,image3,image4,image5,image6,image7,image8,UserID
0,Session_1,58.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,True,False,False,False,False,True,False,False,False,User-1
1,Session_1,62.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,True,False,False,False,False,True,False,False,False,User-1
2,Session_1,66.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,True,False,False,False,False,True,False,False,False,User-1
3,Session_1,70.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,True,False,False,False,False,True,False,False,False,User-1
4,Session_1,74.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,True,False,False,False,False,True,False,False,False,User-1


In [6]:
i = 0
for u in df.UserID.unique():
    i+=1
    print(f"{i}. Sessions for '{u}' are: {df[df.UserID == u].SessionID.nunique()}")

1. Sessions for 'User-1' are: 11
2. Sessions for 'User-2' are: 11
3. Sessions for 'User-3' are: 11
4. Sessions for 'User-4' are: 11
5. Sessions for 'User-5' are: 11
6. Sessions for 'User-6' are: 11
7. Sessions for 'User-7' are: 11
8. Sessions for 'User-8' are: 11
9. Sessions for 'User-9' are: 11
10. Sessions for 'User-10' are: 11
11. Sessions for 'User-11' are: 2
12. Sessions for 'User-12' are: 2
13. Sessions for 'User-13' are: 2
14. Sessions for 'User-14' are: 3
15. Sessions for 'User-15' are: 2
16. Sessions for 'User-16' are: 2


In [7]:
i = 0
totalDuration = 0
for s in df.SessionID.unique():
    i+=1
    sessionDuration = df[df.SessionID == s].StartTime.iloc[-1]-df[df.SessionID == s].StartTime.iloc[0]
    print(f"{i}. Sessions '{s}' has len.: {sessionDuration}")
    totalDuration += sessionDuration
    
print(f"\nTotal Duration for all {i} sessions: {totalDuration}")
print(f"Average Session Duration: {totalDuration/i}")

1. Sessions 'Session_1' has len.: 298.77161
2. Sessions 'Session_2' has len.: 77.79948
3. Sessions 'Session_3' has len.: 83.70704900000001
4. Sessions 'Session_4' has len.: 59.69359
5. Sessions 'Session_5' has len.: 135.71003100000001
6. Sessions 'Session_6' has len.: 157.16160000000002
7. Sessions 'Session_7' has len.: 126.87886
8. Sessions 'Session_8' has len.: 158.27733
9. Sessions 'Session_9' has len.: 90.747097
10. Sessions 'Session_10' has len.: 164.026734
11. Sessions 'Session_11' has len.: 119.01895200000001
12. Sessions 'Session_12' has len.: 175.72571000000002
13. Sessions 'Session_13' has len.: 116.14623999999999
14. Sessions 'Session_14' has len.: 110.40374
15. Sessions 'Session_15' has len.: 148.87786
16. Sessions 'Session_16' has len.: 138.32633
17. Sessions 'Session_17' has len.: 163.95143000000002
18. Sessions 'Session_18' has len.: 96.31892
19. Sessions 'Session_19' has len.: 158.60901
20. Sessions 'Session_20' has len.: 116.52373
21. Sessions 'Session_21' has len.: 14

In [8]:
df.UserID.value_counts()

UserID
User-9     402
User-10    386
User-4     377
User-2     358
User-1     343
User-8     341
User-7     335
User-3     306
User-5     293
User-6     257
User-16    147
User-15    141
User-12    128
User-13    112
User-14    111
User-11     84
Name: count, dtype: int64

In [9]:
df.describe()

,StartTime,TotalTimeSpent,DistanceFromStart,DistanceFromPrevious,DistanceSoFar,AngleChange,bearing_elevation,AngularShift,LinVelocity,Jerk,AngularShiftRate,AngularShiftJerkRate
count,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000
mean,106.304653,3.455959,19.085845,2.763929,52.968930,78.216646,0.084821,0.186593,0.608674,0.010206,0.039902,0.000659
std,71.405322,1.084562,11.208863,7.007967,54.440483,50.319742,0.087991,0.594205,1.700593,0.365398,0.136538,0.056216
min,4.700531,0.005690,2.256282,0.000000,0.000000,0.894374,0.012120,0.000000,0.000000,-4.046348,0.000000,-0.377884
25%,54.022500,3.659550,14.316257,0.000000,12.015596,39.633980,0.043187,0.000000,0.000000,0.000000,0.000000,0.000000
50%,92.186360,4.000000,17.019090,0.000000,45.876409,69.652840,0.063748,0.000000,0.000000,0.000000,0.000000,0.000000
75%,141.063600,4.000000,21.928547,0.000000,70.345007,116.302600,0.092041,0.000000,0.000000,0.065935,0.000000,0.000000
max,479.429100,4.000000,82.670841,97.861518,386.489480,179.916100,0.931293,5.037579,26.445485,4.954454,1.511537,0.723687


In [10]:
df.isna().sum()

SessionID               0
StartTime               0
TotalTimeSpent          0
DistanceFromStart       0
DistanceFromPrevious    0
DistanceSoFar           0
AngleChange             0
bearing_elevation       0
AngularShift            0
LinVelocity             0
Jerk                    0
AngularShiftRate        0
AngularShiftJerkRate    0
Dir_B                   0
Dir_F                   0
Dir_L                   0
Dir_LB                  0
Dir_LF                  0
Dir_R                   0
Dir_RB                  0
Dir_RF                  0
Dir_S                   0
image1                  0
image2                  0
image3                  0
image4                  0
image5                  0
image6                  0
image7                  0
image8                  0
UserID                  0
dtype: int64

#### Label Encoding

In [11]:
le = LabelEncoder()
df['UserID'] = le.fit_transform(df['UserID'])

In [12]:
# Create encoding map
encoding_map = pd.DataFrame({
    'original_label': le.classes_,
    'encoded_label': range(len(le.classes_))
})

In [13]:
df.tail(3)

,SessionID,StartTime,TotalTimeSpent,DistanceFromStart,DistanceFromPrevious,DistanceSoFar,AngleChange,bearing_elevation,AngularShift,LinVelocity,...,Dir_S,image1,image2,image3,image4,image5,image6,image7,image8,UserID
4118,Session_123,337.9767,4.0000,7.072571,0.0,203.911283,62.30661,0.169069,0.0,0.0,...,True,False,False,False,False,False,False,False,True,7
4119,Session_123,341.9767,4.0000,7.072571,0.0,203.911283,62.30661,0.169069,0.0,0.0,...,True,False,False,False,False,False,False,False,True,7
4120,Session_123,345.9767,3.7711,7.072571,0.0,203.911283,62.30661,0.169069,0.0,0.0,...,True,False,False,False,False,False,False,False,True,7


In [14]:
df.head()

,SessionID,StartTime,TotalTimeSpent,DistanceFromStart,DistanceFromPrevious,DistanceSoFar,AngleChange,bearing_elevation,AngularShift,LinVelocity,...,Dir_S,image1,image2,image3,image4,image5,image6,image7,image8,UserID
0,Session_1,58.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,True,False,False,False,False,True,False,False,False,0
1,Session_1,62.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,True,False,False,False,False,True,False,False,False,0
2,Session_1,66.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,True,False,False,False,False,True,False,False,False,0
3,Session_1,70.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,True,False,False,False,False,True,False,False,False,0
4,Session_1,74.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,True,False,False,False,False,True,False,False,False,0


#### OHE _(`bool` to `int`)_

In [15]:
## for later compatibility with model
BOOL_CATG_COLS = list(df.select_dtypes(include='bool').columns)
# Boolean to int conversion
for col in BOOL_CATG_COLS:
    df[col] = df[col].astype(int)

In [16]:
df.head(3)

,SessionID,StartTime,TotalTimeSpent,DistanceFromStart,DistanceFromPrevious,DistanceSoFar,AngleChange,bearing_elevation,AngularShift,LinVelocity,...,Dir_S,image1,image2,image3,image4,image5,image6,image7,image8,UserID
0,Session_1,58.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,1,0,0,0,0,1,0,0,0,0
1,Session_1,62.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,1,0,0,0,0,1,0,0,0,0
2,Session_1,66.83619,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,...,1,0,0,0,0,1,0,0,0,0


### Sampleing Sequences

In [17]:
def build_samples(df, window_size, stride):
    samples = []
    sample_id = 0

    for (UserID, session), g in df.groupby(["UserID", "SessionID"]):
        g = g.sort_values("StartTime")

        for start in range(0, len(g) - window_size + 1, stride):
            window = g.iloc[start:start + window_size].copy()
            window["SampleID"] = sample_id
            samples.append(window)
            sample_id += 1

    return pd.concat(samples, ignore_index=True)

In [18]:
df_samples = build_samples(df, W, STRIDE)
df_samples.shape

(16291, 32)

In [19]:
# dropping cols which are unwnated now
df_samples = df_samples.drop(columns=["StartTime", "SessionID"])
df_samples.shape

(16291, 30)

In [20]:
df_samples.tail()

,TotalTimeSpent,DistanceFromStart,DistanceFromPrevious,DistanceSoFar,AngleChange,bearing_elevation,AngularShift,LinVelocity,Jerk,AngularShiftRate,...,image1,image2,image3,image4,image5,image6,image7,image8,UserID,SampleID
16286,4.0,22.0761,0.0,110.013465,15.54977,0.082331,0.0,0.0,0.544119,0.0,...,0,0,0,0,1,0,0,0,15,1480
16287,4.0,22.0761,0.0,110.013465,15.54977,0.082331,0.0,0.0,0.000000,0.0,...,0,0,0,0,1,0,0,0,15,1480
16288,4.0,22.0761,0.0,110.013465,15.54977,0.082331,0.0,0.0,0.000000,0.0,...,0,0,0,0,1,0,0,0,15,1480
16289,4.0,22.0761,0.0,110.013465,15.54977,0.082331,0.0,0.0,0.000000,0.0,...,0,0,0,0,1,0,0,0,15,1480
16290,4.0,22.0761,0.0,110.013465,15.54977,0.082331,0.0,0.0,0.000000,0.0,...,0,0,0,0,1,0,0,0,15,1480


In [21]:
for u in df.UserID.unique().tolist():
    print(f'Sample counts for {u}: {df_samples[df_samples.UserID == u].SampleID.nunique()}')

Sample counts for 0: 119
Sample counts for 8: 126
Sample counts for 9: 101
Sample counts for 10: 138
Sample counts for 11: 94
Sample counts for 12: 76
Sample counts for 13: 116
Sample counts for 14: 120
Sample counts for 15: 149
Sample counts for 1: 141
Sample counts for 2: 33
Sample counts for 3: 55
Sample counts for 4: 47
Sample counts for 5: 41
Sample counts for 6: 61
Sample counts for 7: 64


---

## Train/Val/Test Split

In [22]:
df_samples.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16291 entries, 0 to 16290
Data columns (total 30 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   TotalTimeSpent        16291 non-null  float64
 1   DistanceFromStart     16291 non-null  float64
 2   DistanceFromPrevious  16291 non-null  float64
 3   DistanceSoFar         16291 non-null  float64
 4   AngleChange           16291 non-null  float64
 5   bearing_elevation     16291 non-null  float64
 6   AngularShift          16291 non-null  float64
 7   LinVelocity           16291 non-null  float64
 8   Jerk                  16291 non-null  float64
 9   AngularShiftRate      16291 non-null  float64
 10  AngularShiftJerkRate  16291 non-null  float64
 11  Dir_B                 16291 non-null  int64  
 12  Dir_F                 16291 non-null  int64  
 13  Dir_L                 16291 non-null  int64  
 14  Dir_LB                16291 non-null  int64  
 15  Dir_LF             

In [23]:
df_samples.SampleID.unique()

array([   0,    1,    2, ..., 1478, 1479, 1480])

In [24]:
sample_ids = df_samples["SampleID"].unique()

In [25]:
# splitting 3 sets
train_ids, temp_ids = train_test_split(
    sample_ids, test_size=0.3, shuffle=True, random_state=42
)

# validation set is smaller then test
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.7, shuffle=True, random_state=42
)

In [26]:
df_train = df_samples[df_samples["SampleID"].isin(train_ids)]
df_val   = df_samples[df_samples["SampleID"].isin(val_ids)]
df_test  = df_samples[df_samples["SampleID"].isin(test_ids)]

df_train.shape, df_val.shape, df_test.shape

((11396, 30), (1463, 30), (3432, 30))

In [27]:
df_train.head()

,TotalTimeSpent,DistanceFromStart,DistanceFromPrevious,DistanceSoFar,AngleChange,bearing_elevation,AngularShift,LinVelocity,Jerk,AngularShiftRate,...,image1,image2,image3,image4,image5,image6,image7,image8,UserID,SampleID
0,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0
1,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0
2,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0
3,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0
4,4.0,2.67886,0.0,0.0,87.62966,0.578041,0.0,0.0,0.0,0.0,...,0,0,0,0,1,0,0,0,0,0


In [28]:
df_val.head()

,TotalTimeSpent,DistanceFromStart,DistanceFromPrevious,DistanceSoFar,AngleChange,bearing_elevation,AngularShift,LinVelocity,Jerk,AngularShiftRate,...,image1,image2,image3,image4,image5,image6,image7,image8,UserID,SampleID
253,4.0000,72.693141,0.000000,190.226046,112.332700,0.026879,0.00000,0.000000,0.000000,0.000000,...,0,0,0,0,0,1,0,0,0,23
254,4.0000,72.693141,0.000000,190.226046,112.332700,0.026879,0.00000,0.000000,0.000000,0.000000,...,0,0,0,0,0,1,0,0,0,23
255,4.0000,72.693141,0.000000,190.226046,112.332700,0.026879,0.00000,0.000000,0.000000,0.000000,...,0,0,0,0,0,1,0,0,0,23
256,2.0339,72.693141,0.000000,190.226046,112.332700,0.026879,0.00000,0.000000,0.000000,0.000000,...,0,0,0,0,0,1,0,0,0,23
257,4.0000,75.899928,8.184885,198.410931,5.698176,0.025737,0.10146,0.916879,0.011506,0.011366,...,0,0,0,0,0,0,1,0,0,23


---

## Normalization and Final Data Prep

### Feature Groups

In [29]:
df[df.columns[:10]].describe()

,StartTime,TotalTimeSpent,DistanceFromStart,DistanceFromPrevious,DistanceSoFar,AngleChange,bearing_elevation,AngularShift,LinVelocity
count,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000,4121.000000
mean,106.304653,3.455959,19.085845,2.763929,52.968930,78.216646,0.084821,0.186593,0.608674
std,71.405322,1.084562,11.208863,7.007967,54.440483,50.319742,0.087991,0.594205,1.700593
min,4.700531,0.005690,2.256282,0.000000,0.000000,0.894374,0.012120,0.000000,0.000000
25%,54.022500,3.659550,14.316257,0.000000,12.015596,39.633980,0.043187,0.000000,0.000000
50%,92.186360,4.000000,17.019090,0.000000,45.876409,69.652840,0.063748,0.000000,0.000000
75%,141.063600,4.000000,21.928547,0.000000,70.345007,116.302600,0.092041,0.000000,0.000000
max,479.429100,4.000000,82.670841,97.861518,386.489480,179.916100,0.931293,5.037579,26.445485


In [30]:
FEATURE_GROUPS = {
    # Non-negative, heavy-tailed
    "log": [
        "DistanceFromStart",
        "DistanceFromPrevious",
        "DistanceSoFar",
        "LinVelocity"
    ],
    
    # Signed, heavy-tailed
    "signed_log": [
        "Jerk",
        "AngularShiftJerkRate"
    ],
    
    # Moderate-scale, continuous
    "zscore": [
        "AngularShift",
        "AngularShiftRate",
        "bearing_elevation",
        "AngleChange"
    ],
    
    # Hard bounded scaling (4 sec max.)
    "minmax_fixed": {
        "TotalTimeSpent": (0.0, 4.0)
    },

    # Binary / categorical (no normalization)
    "categorical": []  # Dir_* and Img_* handled automatically
}

In [31]:
setWiseLen = len(set(FEATURE_GROUPS["log"])) + len(set(FEATURE_GROUPS["signed_log"])) + len(set(FEATURE_GROUPS["zscore"])) + len(set(FEATURE_GROUPS["minmax_fixed"]))
listWiseLen = len(FEATURE_GROUPS["log"]) + len(FEATURE_GROUPS["signed_log"]) + len(FEATURE_GROUPS["zscore"]) + len(FEATURE_GROUPS["minmax_fixed"])
allColCount = df_samples.shape[-1]

setWiseLen, listWiseLen, allColCount

(11, 11, 30)

**all columns are considered**

### Norm.

In [32]:
def compute_norm_stats(
    train_df,
    feature_groups,
    clip_percentile=99.5,
    eps=1e-6
):
    """
    Compute normalization statistics ONLY from training data.
    Returns a dictionary of parameters to be reused later.
    """

    stats = {}

    # ---------- LOG (non-negative) ----------
    stats["log"] = {}
    for col in feature_groups.get("log", []):
        x = train_df[col].values
        clip_max = np.percentile(x, clip_percentile)

        x_clip = np.clip(x, 0.0, clip_max)
        x_log = np.log1p(x_clip)

        stats["log"][col] = {
            "clip_max": clip_max,
            "mean": x_log.mean(),
            "std": x_log.std() + eps
        }

    # ---------- SIGNED LOG ----------
    stats["signed_log"] = {}
    for col in feature_groups.get("signed_log", []):
        x = train_df[col].values
        clip_abs = np.percentile(np.abs(x), clip_percentile)

        x_clip = np.clip(x, -clip_abs, clip_abs)
        x_slog = np.sign(x_clip) * np.log1p(np.abs(x_clip))

        stats["signed_log"][col] = {
            "clip_abs": clip_abs,
            "mean": x_slog.mean(),
            "std": x_slog.std() + eps
        }

    # ---------- Z-SCORE ----------
    stats["zscore"] = {}
    for col in feature_groups.get("zscore", []):
        x = train_df[col].values
        stats["zscore"][col] = {
            "mean": x.mean(),
            "std": x.std() + eps
        }

    #----------- MIN-MAX ----------
    stats["minmax_fixed"] = FEATURE_GROUPS["minmax_fixed"]

    return stats

---

In [33]:
def apply_normalization(df, norm_stats):
    """
    Apply previously computed normalization stats to a dataframe.
    Does NOT recompute statistics.
    """

    df = df.copy()

    # ---------- LOG ----------
    for col, p in norm_stats.get("log", {}).items():
        x = df[col].values
        x = np.clip(x, 0.0, p["clip_max"])
        x = np.log1p(x)
        df[col] = (x - p["mean"]) / p["std"]

    # ---------- SIGNED LOG ----------
    for col, p in norm_stats.get("signed_log", {}).items():
        x = df[col].values
        x = np.clip(x, -p["clip_abs"], p["clip_abs"])
        x = np.sign(x) * np.log1p(np.abs(x))
        df[col] = (x - p["mean"]) / p["std"]

    # ---------- Z-SCORE ----------
    for col, p in norm_stats.get("zscore", {}).items():
        df[col] = (df[col] - p["mean"]) / p["std"]

    # ---------- MIN–MAX (FIXED RANGE) ----------
    for col, (mn, mx) in norm_stats.get("minmax_fixed", {}).items():
        df[col] = (df[col] - mn) / (mx - mn)

    # ---------- categorical: untouched ----------
    return df


---

#### Applying

In [34]:
# 1. Fit on TRAIN ONLY
norm_stats = compute_norm_stats(df_train, FEATURE_GROUPS)
norm_stats

{'log': {'DistanceFromStart': {'clip_max': np.float64(82.67084143946433),
   'mean': np.float64(2.909970212503244),
   'std': np.float64(0.47924464359184543)},
  'DistanceFromPrevious': {'clip_max': np.float64(39.01552811538221),
   'mean': np.float64(0.5441878379550215),
   'std': np.float64(1.0456302985519914)},
  'DistanceSoFar': {'clip_max': np.float64(299.60938051433095),
   'mean': np.float64(3.482523309920158),
   'std': np.float64(1.3006913608920592)},
  'LinVelocity': {'clip_max': np.float64(11.042939111598702),
   'mean': np.float64(0.26525512428314985),
   'std': np.float64(0.5547681619364966)}},
 'signed_log': {'Jerk': {'clip_abs': np.float64(2.1746614877472354),
   'mean': np.float64(0.011239233751075259),
   'std': np.float64(0.24500868957912392)},
  'AngularShiftJerkRate': {'clip_abs': np.float64(0.344784444854079),
   'mean': np.float64(0.00014049468152180252),
   'std': np.float64(0.0502020059806234)}},
 'zscore': {'AngularShift': {'mean': np.float64(0.1991786919408802

In [35]:
# 2. Apply everywhere
df_train_norm = apply_normalization(df_train, norm_stats)
df_val_norm   = apply_normalization(df_val, norm_stats)
df_test_norm  = apply_normalization(df_test, norm_stats)

df_train_norm.shape, df_val_norm.shape, df_test_norm.shape

((11396, 30), (1463, 30), (3432, 30))

---

In [36]:
df_train_norm.head()

,TotalTimeSpent,DistanceFromStart,DistanceFromPrevious,DistanceSoFar,AngleChange,bearing_elevation,AngularShift,LinVelocity,Jerk,AngularShiftRate,...,image1,image2,image3,image4,image5,image6,image7,image8,UserID,SampleID
0,1.0,-3.35396,-0.52044,-2.67744,0.243975,5.370798,-0.317283,-0.478137,-0.045873,-0.298153,...,0,0,0,0,1,0,0,0,0,0
1,1.0,-3.35396,-0.52044,-2.67744,0.243975,5.370798,-0.317283,-0.478137,-0.045873,-0.298153,...,0,0,0,0,1,0,0,0,0,0
2,1.0,-3.35396,-0.52044,-2.67744,0.243975,5.370798,-0.317283,-0.478137,-0.045873,-0.298153,...,0,0,0,0,1,0,0,0,0,0
3,1.0,-3.35396,-0.52044,-2.67744,0.243975,5.370798,-0.317283,-0.478137,-0.045873,-0.298153,...,0,0,0,0,1,0,0,0,0,0
4,1.0,-3.35396,-0.52044,-2.67744,0.243975,5.370798,-0.317283,-0.478137,-0.045873,-0.298153,...,0,0,0,0,1,0,0,0,0,0


In [37]:
df_val_norm.head()

,TotalTimeSpent,DistanceFromStart,DistanceFromPrevious,DistanceSoFar,AngleChange,bearing_elevation,AngularShift,LinVelocity,Jerk,AngularShiftRate,...,image1,image2,image3,image4,image5,image6,image7,image8,UserID,SampleID
253,1.000000,2.900271,-0.520440,1.361532,0.746992,-0.630397,-0.317283,-0.478137,-0.045873,-0.298153,...,0,0,0,0,0,1,0,0,0,23
254,1.000000,2.900271,-0.520440,1.361532,0.746992,-0.630397,-0.317283,-0.478137,-0.045873,-0.298153,...,0,0,0,0,0,1,0,0,0,23
255,1.000000,2.900271,-0.520440,1.361532,0.746992,-0.630397,-0.317283,-0.478137,-0.045873,-0.298153,...,0,0,0,0,0,1,0,0,0,23
256,0.508475,2.900271,-0.520440,1.361532,0.746992,-0.630397,-0.317283,-0.478137,-0.045873,-0.298153,...,0,0,0,0,0,1,0,0,0,23
257,1.000000,2.989151,1.600347,1.393754,-1.424360,-0.642833,-0.155661,0.694782,0.000819,-0.219561,...,0,0,0,0,0,0,1,0,0,23


In [38]:
df_test_norm.head()

,TotalTimeSpent,DistanceFromStart,DistanceFromPrevious,DistanceSoFar,AngleChange,bearing_elevation,AngularShift,LinVelocity,Jerk,AngularShiftRate,...,image1,image2,image3,image4,image5,image6,image7,image8,UserID,SampleID
110,1.0,0.092076,2.927697,0.695883,0.09287,0.263495,4.567505,4.007482,4.617749,7.569355,...,0,0,0,0,0,1,0,0,0,10
111,1.0,0.092076,-0.520440,0.695883,0.09287,0.263495,-0.317283,-0.478137,-4.613630,-0.298153,...,0,0,0,0,0,1,0,0,0,10
112,1.0,0.092076,-0.520440,0.695883,0.09287,0.263495,-0.317283,-0.478137,2.420995,-0.298153,...,0,0,0,0,0,1,0,0,0,10
113,1.0,0.092076,-0.520440,0.695883,0.09287,0.263495,-0.317283,-0.478137,-0.045873,-0.298153,...,0,0,0,0,0,1,0,0,0,10
114,1.0,0.092076,-0.520440,0.695883,0.09287,0.263495,-0.317283,-0.478137,-0.045873,-0.298153,...,0,0,0,0,0,1,0,0,0,10


## Exporting

In [39]:
# Exporting heater labels
encoding_map.to_csv(f'{MODEL_DATA_PATH}raw_split/category_label_map.csv', index=False)

In [40]:
## normalizers
with open(f"{MODEL_PATH}raw_normalization_stats.pkl", "wb") as f:
    pickle.dump(norm_stats, f)

In [ ]:
## csv version of datasets
df_train_norm.to_csv(f"{MODEL_DATA_PATH}/train_normalized.csv", index=False)
df_val_norm.to_csv(f"{MODEL_DATA_PATH}/val_normalized.csv", index=False)
df_test_norm.to_csv(f"{MODEL_DATA_PATH}/test_normalized.csv", index=False)

---  
---  